##### RV Plotted Against IV

In [48]:
import numpy as np
import pandas as pd
import os
import plotly.graph_objects as go
from datetime import date

spy_csv_dir = "~/python-projects/ed-backtest/backtester/data/sp_constituents/SPY.csv"
vix_csv_dir = "~/python-projects/ed-backtest/backtester/data/sp_constituents/^VIX.csv"

spy_df = pd.read_csv(os.path.expanduser(spy_csv_dir), parse_dates=['Date'])
vix_df = pd.read_csv(os.path.expanduser(vix_csv_dir), parse_dates=['Date'])

spy_df['Date'] = pd.to_datetime(spy_df['Date'], utc=True).dt.tz_localize(None).dt.normalize()
vix_df['Date'] = pd.to_datetime(vix_df['Date'], utc=True).dt.tz_localize(None).dt.normalize()

spy_df['Log_Returns'] = np.log(spy_df['Close'] / spy_df['Close'].shift(1))

window = 21
spy_df['RV'] = spy_df['Log_Returns'].rolling(window).std() * np.sqrt(252) * 100 #IV reflects 30 day options, so rolling 30d is used
merge_df = pd.merge(spy_df[['Date', 'RV']], vix_df[['Date', 'Close']], on='Date', how='inner', suffixes=('', '_VIX'))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=merge_df['Date'], y=merge_df['RV'], name=f"RV: {window}d"
))
fig.add_trace(go.Scatter(
    x=merge_df['Date'], y=merge_df['Close'], name="IV"
))
fig.show()


##### yfinance News Collection

In [25]:
import yfinance as yf

news = yf.Search("^TNX", news_count=11).news

print(len(news))

for s in news[:10]:
    if "^TNX" in s.get("relatedTickers", ["^TNX"]):
        print(s['providerPublishTime'], s['title'])




10
1791794512 Is 8% next? Why mortgage rates have surged, and may stay higher for longer
1791722203 Bond yields are surging worldwide — and France is on the 'ugly side': Chart of the Day
1791719856 Bubble-like stock valuations and debt crisis chatter aren't exactly bullish
1791706172 General American Investors Company And 2 More Top Dividend Stocks
1791627129 Microsoft Plus 2 Elite Defensive Stocks
1791554057 Tech rebounds as AI trade jitters ease: AlphaCheck
1791478815 Oil falls to $103 as Trump says US will not attack Iran before the midterm elections
1791475730 Mortgage rates are stuck at the highest level in nearly three years: Mortgage and refinance interest rates today
1791472040 Rapid bond yield moves have preceded nearly every major financial blowup: Chart of the Day
1791468275 Chip stocks retreat despite Samsung's record profit guidance: AlphaCheck


##### LECTURE 3: Regression with single regressor

In [47]:
import pandas as pd
import os
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm


csv_dir = "~/BAYES/Data_Analysis/data_sets/class.csv"
path = os.path.expanduser(csv_dir)


df = pd.read_csv(path)
print(df.head())
#print(f"\n\n{df.columns}")


model = LinearRegression()

y = df['SALARY']
X = df[['ROE']]



model.fit(X, y)

#print(f"Statsmodels R-Square: {sm_results.t_test([1, 0])}")
n = len(df)
p = X.shape[1]

r2 = model.score(X, y)
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"\n\nR-Square: {(r2 * 100):.4f}")
print(f"\nAdj. R-Square: {adj_r2:.4f}")
print(f"\nBeta 2: {model.coef_[0]:.4f}")



   SALARY   ROE    ROS    SALES  INDUS  UTILITY  PCROE  PCSALARY
0    1095  14.1  191.0  27595.0      1        0  106.4        20
1    1001  10.9   13.0   9958.0      1        0  -30.6        32
2    1122  23.5   14.0   6125.9      1        0  -16.3         9
3     578   5.9  -21.0  16246.0      1        0  -25.7        -9
4    1368  13.8   56.0  21783.2      1        0   -3.0         7


R-Square: 1.3189

Adj. R-Square: 0.0084

Beta 2: 18.5012


#### TUTORIAL 3: Hypothesis Testing and Restrictions

In [43]:
import scipy.stats as st
import statsmodels.formula.api as smf

#print(df.columns)

# F-test critical value: Hypothesis test with restriction
"""
An F-test checks whether J restrictions hold at once, for example:

H0: B3 = 0

1: Fit the unrestriced model (all variables) and the restricted model (B2 and B3 dropped)
2: Compute F = ((SSE_r - SSE_u) / J) / (SSE_u / (T - k))
3: Compare with the critical value

If F-stat > F_crit: reject H0. The restrictions don't hold.

If F-stat < F_crit: fail to reject.


"""
# Align number of observations
df = df.dropna(subset=['SALARY', 'ROE', 'ROS'])


# Step 1: Fit both the models
u_results = smf.ols('SALARY ~ ROE + ROS', data=df).fit() # 2 Regressors: B1 = Intercept, B2 = ROE, B3 = ROS
r_results = smf.ols('SALARY ~ ROE', data=df).fit() # Restricted model: ROS = 0


# Step 2: Compute the F statistic
SSE_u = u_results.ssr
SSE_r = r_results.ssr

T, k, J = int(u_results.nobs), len(u_results.params), 1 # Where T = No. observations, k = No. parameters in unrestricted model, J = No. restrictions tested jointly 


F = ((SSE_r - SSE_u) / J) / (SSE_u / (T - k)) # Manual computation of F-test
F_crit = st.f.ppf(0.95, J, T-k)
p_val = st.f.sf(F, J, T-k)

print(f"Critical Value: {F_crit} | P-value: {p_val}")
print(f'\nCalculated F: {F} | SM-Generated F: {u_results.f_test('ROS = 0')}') # Automatic computation of F-stat
auto_F = u_results.f_test('ROS = 0')

# Step 3: Compare with critical value
if auto_F.fvalue > F_crit: # Access the fvalue through .fvalue
    print(f"Reject Null")
elif auto_F.fvalue < F_crit:
    print(f"Fail to reject Null")



# Output regression results
print(f"\n{u_results.summary()}")

# Individual outputs can still be accessed as such
sm_rsq = u_results.rsquared
sm_adj_r = u_results.rsquared_adj















Critical Value: 3.8874466631593303 | P-value: 0.32344266115643466

Calculated F: 0.979711205752902 | SM-Generated F: <F test: F=0.9797112057529259, p=0.32344266115642856, df_denom=204, df_num=1>
Fail to reject Null

                            OLS Regression Results                            
Dep. Variable:                 SALARY   R-squared:                       0.018
Model:                            OLS   Adj. R-squared:                  0.008
Method:                 Least Squares   F-statistic:                     1.851
Date:                Tue, 20 Oct 2026   Prob (F-statistic):              0.160
Time:                        23:18:28   Log-Likelihood:                -1787.7
No. Observations:                 207   AIC:                             3581.
Df Residuals:                     204   BIC:                             3591.
Df Model:                           2                                         
Covariance Type:            nonrobust                                    

#### JPMC PRACTICE QUESTION: 

In [ ]:
def FizzBuzz(n):
    if n % 3 == 0 and n % 5 == 0:
        print("FizzBuzz")

    elif n % 3 == 0 and n % 5 != 0:
        print("Fizz")

    elif n % 5 == 0 and n % 3 != 0:
        print("Buzz")

    else:
        print(n)






FizzBuzz(2086)

2086


#### Securities & Investments - Data Download Code

In [ ]:
import pandas as pd
import yfinance as yf
 
# ---------------- Settings ----------------
TICKERS = [
    "JNJ", "UNH", "AZN", "PG", "KO", "RACE", "DIS", "V",
    "JPM", "LMT", "HD", "ORCL", "CRM", "MCD", "XOM",
]
START = "2020-12-31"   # change as needed
END = "2026-02-01"             # None = today
OUTPUT_FILE = "monthly_returns_pt2.xlsx"
# ------------------------------------------
 
# 1. Download daily prices (auto_adjust=True -> prices include dividends/splits)
raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=False)
prices = raw["Close"].dropna(how="all")
 
# Warn about any tickers that returned no data
missing = [t for t in TICKERS if t not in prices.columns or prices[t].dropna().empty]
if missing:
    print(f"Warning: no data returned for: {missing}")
 
# Keep the original ticker order
prices = prices[[t for t in TICKERS if t in prices.columns]]
 
# 2. Convert to month-end prices and compute monthly returns
try:
    monthly_prices = prices.resample("ME").last()   # pandas >= 2.2
except ValueError:
    monthly_prices = prices.resample("M").last()    # older pandas
 
monthly_returns = monthly_prices.pct_change(fill_method=None).iloc[1:]
 
# Use a clean month label as the index (e.g. 2024-03)
monthly_returns.index = monthly_returns.index.strftime("%Y-%m")
monthly_returns.index.name = "Month"
monthly_prices.index = monthly_prices.index.strftime("%Y-%m")
monthly_prices.index.name = "Month"
 
# 3. Summary statistics
summary = pd.DataFrame({
    "Mean monthly return": monthly_returns.mean(),
    "Monthly volatility": monthly_returns.std(),
    "Annualised return (geometric)": (1 + monthly_returns).prod() ** (12 / monthly_returns.count()) - 1,
    "Annualised volatility": monthly_returns.std() * (12 ** 0.5),
    "Best month": monthly_returns.max(),
    "Worst month": monthly_returns.min(),
    "Months of data": monthly_returns.count(),
})
 
# 4. Write to Excel
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    monthly_returns.to_excel(writer, sheet_name="Monthly Returns")
    summary.to_excel(writer, sheet_name="Summary")
    monthly_prices.to_excel(writer, sheet_name="Month-end Prices")
 
    # Formatting
    ws = writer.sheets["Monthly Returns"]
    for row in ws.iter_rows(min_row=2, min_col=2):
        for cell in row:
            cell.number_format = "0.00%"
 
    ws = writer.sheets["Summary"]
    for row in ws.iter_rows(min_row=2, min_col=2, max_col=7):
        for cell in row:
            cell.number_format = "0.00%"
 
    ws = writer.sheets["Month-end Prices"]
    for row in ws.iter_rows(min_row=2, min_col=2):
        for cell in row:
            cell.number_format = "#,##0.00"
 
    for ws in writer.book.worksheets:
        ws.freeze_panes = "B2"
        ws.column_dimensions["A"].width = 30 if ws.title == "Summary" else 12
        for col in ws.iter_cols(min_col=2):
            ws.column_dimensions[col[0].column_letter].width = 16
 
print(f"Saved {OUTPUT_FILE}")
 


##### Data Analysis Tutorial 4: Tesla Stock and Market Portfolio Comparison

In [ ]:
import pandas as pd
import os


csv_dir = "~/python-projects/ed-backtest/backtester/data/classwork_sets/tesla_class.csv"
path = os.path.expanduser(csv_dir)
df = pd.read_csv(path, parse_dates=["Date"])
#print(df.columns)

df['TSLA_Returns'] = df['TSLA'].pct_change() # Returns of TSLA
df['MKT_Returns'] = df['MKT'].pct_change() # Returns of Market
df['TBILL3M'] = df['TBILL3M'] / 100 # Convert to pct
df['TBILL_DeAnn'] = (1 + df['TBILL3M'])**(1/12) - 1 # Deannualise T-Bill to monthly (1/12)
df['TSLA_Excess_Returns'] = df['TSLA_Returns'] - df['TBILL_DeAnn'] # TSLA excess returns
df['MKT_Excess_Returns'] = df['MKT_Returns'] - df['TBILL_DeAnn'] # MKT excess returns


0     2010-07-31
1     2010-08-31
2     2010-09-30
3     2010-10-31
4     2010-11-30
         ...    
189   2026-04-30
190   2026-05-31
191   2026-06-30
192   2026-07-31
193   2026-08-31
Name: Date, Length: 194, dtype: datetime64[us]


##### Summarise the Excess Returns of TSLA and MKT

In [25]:
import numpy as np

tsla_mean = df['TSLA_Excess_Returns'].mean() # 0.0425805555457681
tsla_med = df['TSLA_Excess_Returns'].median() # 0.01099262108482768

"""
The mean of TSLA returns is far greater than that of the median. This suggests there is significant skewness in the distribution of the returns.
In particular, the returns of TSLA are skewed right (positive skew), this means that there are fat tails present which are pushing the mean upwards.

"""

mkt_mean = df['MKT_Excess_Returns'].mean() # 0.009700739930258892
mkt_med = df['MKT_Excess_Returns'].median() # 0.01561964862093368

"""
The mean of the excess MKT returns is lower than the median, this tells us that there is a negative skew of the returns.

On average: The MKT provided investors with a higher return (higher median), however using solely the mean would tell a different story.
"""


'\nThe mean of the excess MKT returns is lower than the median, this tells us that there is a negative skew of the returns.\n\nOn average: The MKT provided investors with a higher return (higher median), however using solely the mean would tell a different story.\n'

##### Plot a Histogram with the Kernel Density Estimate to Approximate the Probability Distribution of TSLA Excess Returns

In [45]:
import plotly.graph_objects as go
from scipy.stats import gaussian_kde
import numpy as np

df = df.dropna()

tsla_kde = gaussian_kde(np.asarray(df['TSLA_Excess_Returns'], dtype="float"))
mkt_kde = gaussian_kde(np.asarray(df['MKT_Excess_Returns']))

tsla_x = np.linspace(df['TSLA_Excess_Returns'].min(), df['TSLA_Excess_Returns'].min(), 200)
mkt_x = np.linspace(df['MKT_Excess_Returns'].min(), df['MKT_Excess_Returns'].min(), 200)



fig = go.Figure()
fig.add_trace(go.Histogram(
    x=df['Date'],
    y=df['TSLA_Excess_Returns'],
    name="TSLA"

))

fig.add_trace(go.scatter(
    x=tsla_x, 
    y=tsla_kde(tsla_x), 
    mode="lines", 
    name="kde"))


fig2 = go.Figure()
fig2.add_trace(go.Histogram(
    x=df['Date'],
    y=df['MKT_Excess_Returns'],
    name="MKT"
))

fig.add_trace(go.Histogram(
    x=df['Date'],
    y=mkt_kde(mkt_x),
    mode="lines",
    name="kde"
))




TypeError: 'module' object is not callable